# Xuất các mô hình ứng viên để đo tốc độ trên Jetson Nano
ONNX (định dạng trao đổi mô hình) giữ FP32 (số thực 32 bit), opset (phiên bản tập toán tử) 12, batch (số ảnh mỗi lô) 1 và kích thước cố định. FPS là số khung hình mỗi giây; notebook chỉ đếm tham số, GFLOPs (tỉ phép toán) và dung lượng file. Thời gian thật phải đo bằng `trtexec` trên xe.

Chỉ dùng **trọng số thử tốc độ**, không dùng để nhận dạng hay điều khiển. `FAKE_DATA=True` dùng khởi tạo ngẫu nhiên cho cả YOLO (mạng phát hiện vật thể) và mô hình đường, chạy CPU (bộ xử lý trung tâm), không tải trọng số COCO (bộ dữ liệu tiền huấn luyện). `False` tải trọng số COCO cho YOLO; mô hình đường vẫn ngẫu nhiên. Cả hai chế độ dùng 20 ảnh tổng hợp để đối chiếu số học, không kiểm chất lượng nhận dạng.

Trước khi chạy trên Colab, chép cấu trúc `round3_final/` vào `MyDrive/UIT_Car_Racing/Vong3/source/`, chỉ cần: `tools/{fps_candidates,export_onnx,compare_outputs,lane_models}.py` và `notebooks/road_train_car.ipynb`. Không cần khoá API (giao diện lập trình). Trên laptop, đặt `SOURCE_ROOT` đến thư mục gốc kho và `LOCAL_ROOT` đến nơi đồng bộ Drive. Chỉ nạp mã nguồn/trọng số tin cậy.

In [ ]:
from pathlib import Path

OUTPUT_DRIVE = "MyDrive/UIT_Car_Racing/Vong3/candidates"
FAKE_DATA = True
YOLO_CANDIDATES = [("yolov8n", 320), ("yolov8n", 416), ("yolo11n", 320), ("yolo11n", 416)]
ROAD_CANDIDATES = ["nanounet", "unet_b_minus"]
SOURCE_ROOT = "/content/drive/MyDrive/UIT_Car_Racing/Vong3/source"
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3/candidates"
INSTALL_DEPS = True
SEED = 42
EXPORT_TIMEOUT_SECONDS = 900

## Môi trường và nguồn mã
Chọn phiên Colab CPU mới và chạy từ trên xuống; cấp quyền kết nối Drive khi được hỏi. Ô cài đặt dùng các thư viện của bộ xuất hiện có, có giới hạn 300 giây. Đã cài đủ thư viện thì đặt `INSTALL_DEPS=False`. Mỗi lần chạy tạo thư mục thời gian riêng dưới `candidates/`; không ghi đè kết quả cũ.

In [ ]:
import os, sys, subprocess, importlib.util, datetime, csv

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "ultralytics==8.4.161",
                    "numpy", "opencv-python", "onnx", "onnxruntime", "onnxsim", "thop"],
                   check=True, timeout=300)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive") / OUTPUT_DRIVE
else:
    OUTPUT_ROOT = Path(LOCAL_ROOT).expanduser().resolve()
SOURCE = Path(SOURCE_ROOT).expanduser().resolve() / "round3_final"
TOOLS = SOURCE / "tools"
required = [TOOLS / (name + ".py") for name in
            ("fps_candidates", "export_onnx", "compare_outputs", "lane_models")]
required += [SOURCE / "notebooks" / "road_train_car.ipynb"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Chép đủ nguồn mã vào SOURCE_ROOT: " + ", ".join(missing))
sys.path.insert(0, str(TOOLS))
from fps_candidates import road_input_hw, candidate_specs
from compare_outputs import read_json, write_image

ROAD_HW = road_input_hw(required[-1])
CANDIDATES = candidate_specs(ROAD_HW, YOLO_CANDIDATES, ROAD_CANDIDATES)
RUN = OUTPUT_ROOT / ("fake" if FAKE_DATA else "coco") / datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN.mkdir(parents=True, exist_ok=False)
os.environ["YOLO_CONFIG_DIR"] = str(RUN / "cache" / "ultralytics")
os.environ["TORCH_HOME"] = str(RUN / "cache" / "torch")
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["OMP_NUM_THREADS"] = "2"
print("Kích thước đường đọc từ notebook huấn luyện:", ROAD_HW)
print("Kết quả:", RUN)

## Ảnh đối chiếu tổng hợp
Tạo 20 ảnh khác nhau với cùng seed (giá trị khởi tạo ngẫu nhiên). Đây là kiểm tra số học giữa PyTorch và ONNX Runtime (bộ thực thi ONNX), không thay thế đối chiếu trên ảnh camera thật trước khi tích hợp điều khiển.

In [ ]:
import numpy as np

IMAGES = RUN / "check_images"
IMAGES.mkdir()
rng = np.random.default_rng(SEED)
for index in range(20):
    h, w = ROAD_HW
    image = rng.integers(0, 256, size=(2*h + index, 2*w + index, 3), dtype=np.uint8)
    write_image(IMAGES / ("{:02d}.png".format(index)), image)

## Xuất và đối chiếu từng ứng viên
Mỗi ứng viên chạy trong tiến trình riêng, tối đa 900 giây mặc định, ghi đầy đủ vào `<tên>_export.log`. Khi lỗi hoặc hết giờ, dừng để xem nhật ký; không bỏ qua ứng viên hỏng. Dùng lại `export_onnx.py`: rút gọn đồ thị, kiểm opset/kích thước, đối chiếu 20 ảnh với sai số xác suất < 0,001; YOLO thêm sai số toạ độ < 0,01 pixel, không có NMS (loại khung trùng) trong mạng.

`_info.json` giữ hợp đồng RGB (đỏ, lục, lam), NCHW (lô, kênh, cao, rộng), float32 /255 ngoài mạng. `weights_purpose="speed_test_only"`; `random_weights` phân biệt trọng số ngẫu nhiên với COCO. `fake_data=true` vì ảnh kiểm tra luôn là tổng hợp. Ngưỡng nhận dạng, hàng quét, nguồn dữ liệu thi và thời gian Nano không áp dụng nên là `null`. Các file này chỉ để đo tốc độ; công cụ nhận dạng cần thông tin/ngưỡng từ mô hình đã huấn luyện thật.

In [ ]:
for spec in CANDIDATES:
    output = RUN / (spec["name"] + ".onnx")
    command = [sys.executable, str(TOOLS / "fps_candidates.py"), spec["architecture"],
               "--size", *map(str, spec["input_hw"]), "--output", str(output),
               "--images", str(IMAGES), "--seed", str(SEED)]
    if FAKE_DATA:
        command.append("--fake-data")
    log = RUN / (spec["name"] + "_export.log")
    with log.open("x", encoding="utf-8") as stream:
        try:
            subprocess.run(command, cwd=RUN, stdout=stream, stderr=subprocess.STDOUT,
                           check=True, timeout=EXPORT_TIMEOUT_SECONDS)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
            raise RuntimeError("Xuất thất bại/hết giờ; xem " + str(log)) from exc
    print("Đã xuất và đối chiếu:", spec["name"])

## Bảng kết quả đếm được
GFLOPs dùng quy ước một phép nhân cộng bằng hai phép toán. Nếu thư viện không đếm được thì để `null`, không suy ra thời gian trên Nano. Bảng cũng được lưu thành `candidates.csv` trong Drive. Chép sáu file `.onnx` cùng `_info.json` lên xe; các lệnh build (biên dịch tối ưu) engine (bản thực thi dành cho đúng GPU của xe) FP16 (số thực 16 bit) và bảng đo nằm ở `round3_final/docs/model_fps.md`.

In [ ]:
rows = []
for spec in CANDIDATES:
    output = RUN / (spec["name"] + ".onnx")
    info = read_json(output.with_name(output.stem + "_info.json"))
    rows.append(dict(model=spec["name"], input_hw="x".join(map(str, info["input_hw"])),
                     parameters=info["parameters"], gflops=info["gflops"],
                     file_bytes=output.stat().st_size, drive_path=str(output)))
with (RUN / "candidates.csv").open("x", encoding="utf-8-sig", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("| Tên | Cao x rộng | Tham số | GFLOPs | Byte | Đường dẫn Drive |")
print("|---|---|---|---|---|---|")
for row in rows:
    print("| " + " | ".join("null" if value is None else str(value) for value in row.values()) + " |")